# Filter country polygons from division areas

Scan the `type=division_area` GeoParquet files, keep features marked as land countries, and export them to `data/results/country_area.parquet`.

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    for parent in (start, *start.parents):
        if (parent / '.git').exists():
            return parent
    raise RuntimeError(f'Could not find repository root from {start}')

REPO_ROOT = find_repo_root(Path.cwd().resolve())
DATA_DIR = REPO_ROOT / 'data'
RESULTS_DIR = DATA_DIR / 'results'
GIS_DIR = REPO_ROOT / 'gis_data'
MODULE_ROOT = REPO_ROOT / 'code' / 'overture_analysis'

if str(MODULE_ROOT) not in sys.path:
    sys.path.append(str(MODULE_ROOT))

RESULTS_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import duckdb

DIVISION_RELEASE = '2025-10-22.0'
DIVISION_AREA_DIR = GIS_DIR / 'overturemaps-us-west-2' / 'release' / DIVISION_RELEASE / 'theme=divisions' / 'type=division_area'
SOURCE_PATTERN = str(DIVISION_AREA_DIR / '*.parquet')
print(f'Using division area data from {DIVISION_AREA_DIR}')

def export_divisions(subtype: str, output_name: str, country: str | None = None):
    filters = ['subtype = ?', 'is_land']
    params = [subtype]
    if country:
        filters.insert(0, 'country = ?')
        params.insert(0, country)
    where_clause = ' AND '.join(filters)
    query = f"""
    SELECT *
    FROM read_parquet(?)
    WHERE {where_clause}
    ORDER BY id
    """
    con = duckdb.connect(database=':memory:')
    try:
        df = con.execute(query, [SOURCE_PATTERN, *params]).fetchdf()
    finally:
        con.close()
    output_path = RESULTS_DIR / output_name
    df.to_parquet(output_path, index=False)
    print(f"Retrieved {len(df)} rows for {subtype} -> {output_path}")
    return df

country_df = export_divisions('country', f'country_area-{DIVISION_RELEASE}.parquet')


In [ ]:
region_df = export_divisions('region', f'region_area-{DIVISION_RELEASE}.parquet')


Save regions by country

In [ ]:
regions_il_df = export_divisions('region', 'regions_IL.parquet', country='IL')
